
<h3 align="center">SA4110 MACHINE LEARNING APPLICATION DEVELOPMENT</h3>
<h4 align="center">CA - GROUP 3 - IMAGE CLASSIFIER</h4>
<hr>

<div class="alert alert-block alert-info">
<b><u>Tasks:</u></b> 
<ol>
<li>Create an Image Classifier (CNN model) to classify images of fruits correctly.</li>
<li>A Fruits Dataset is provided that consists of these 3 Classes: -</li>
    <ul>
    <li>Apple</li>
    <li>Banana</li>
    <li>Orange</li>
    </ul>
<li>Use the Images in train.zip and test.zip to Train and Test your Image Classifier.</li>
<li>Document your experiments and results in improving your model’s accuracy.</li>
<li>The Following Activities can Improve your Model’s Accuracy: -</li>
    <ul>
    <li>Balance out the Number of Samples in Each Class</li>
    <li>Correct any mis-labelling in any of the 3 Classes</li>
    <li>Image Augmentation to Generate more Data </li>
    </ul>
<li>Use Matplotlib to Generate any Plots that can Help the Reader understand your Work Better.</li>
</ol></div>



In [2]:
# ========================================================== #
# 1. (Optional) Verification of Base Libraries / Packages 
# Purpose: Verify that the host environment has the necessary
# packages or libraries required to run the script.
# ========================================================== #

import sys
import subprocess
import importlib.util

def verify_and_install():
    required_packages = {
        'pandas': 'pandas',
        'numpy': 'numpy',
        'matplotlib': 'matplotlib',
        'seaborn': 'seaborn',
        'IPython': 'ipython',
        'PIL': 'Pillow',
        'torch': 'torch',
        'torchvision': 'torchvision',
        'keras': 'keras',
        'sklearn': 'scikit-learn'
    }

    print("--- Verifying Environment Setup ---")
    
    for import_name, pip_name in required_packages.items():
        # Check if the module can be found in the host environment
        if importlib.util.find_spec(import_name) is None:
            print(f"❌ '{import_name}' is missing. Installing '{pip_name}'...")
            try:
                # Run pip install securely via the current python executable
                subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
                print(f"✅ Successfully installed '{pip_name}'.")
            except subprocess.CalledProcessError as e:
                # Stop the script if a critical package fails to install
                print(f"⚠️ Failed to install '{pip_name}'. Error: {e}")
                sys.exit(1) 
        else:
            print(f"✅ '{import_name}' is already installed.")
            
    print("-" * 70)
    print("All necessary packages are ready. Proceeding with the script...")
    print("-" * 70 + "\n")

# Run the verification before importing the third-party libraries
verify_and_install()

--- Verifying Environment Setup ---
✅ 'pandas' is already installed.
✅ 'numpy' is already installed.
✅ 'matplotlib' is already installed.
✅ 'seaborn' is already installed.
✅ 'IPython' is already installed.
✅ 'PIL' is already installed.
✅ 'torch' is already installed.
✅ 'torchvision' is already installed.
✅ 'keras' is already installed.
✅ 'sklearn' is already installed.
----------------------------------------------------------------------
All necessary packages are ready. Proceeding with the script...
----------------------------------------------------------------------



In [3]:
# ========================================================== #
# 2. Importing the Required Libraries and Packages
# Purpose: Import the Libraries and Packages to the Host
# Environment for Data Manipulation, Visualization, Image Processing,
# Deep Learning, and Machine Learning Utilities.
# ========================================================== #

# A. System and OS-level operations
import os
import statistics
from pathlib import Path
os.environ["KERAS_BACKEND"] = "torch"

# B. Data Manipulation and Mathematical Operations
import pandas as pd
import numpy as np

# C. Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# D. Image Processing
from PIL import Image

# E. Deep Learning Frameworks (PyTorch & Keras)
import torch
import keras
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# F. Machine Learning Utilities and Metrics (Scikit-Learn)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report


In [17]:
# ========================================================== #
# 3. Load and Process the Image Data from the Directories
# Purpose: Scan the 'train' and 'test' directories for .jpg 
# images, extract class labels from filenames, calculate 
# median dimensions, and compile the data into DataFrames 
# for prior for model training and evaluation.
# ========================================================== #

def process_image_directory(directory_path, dataset_name="Dataset"):
    
    log_html = f"<h4>Processing Status</h4>"
    log_html += f"<ul style='text-align: left;'>"
    
    file_names = []
    class_labels = []
    widths = []
    heights = []
    error_logs = []

    # Single pass through the directory
    for img_path in directory_path.glob("*.jpg"):
        try:
            # A. Open a valid data image file and extract its dimensions
            with Image.open(img_path) as img:
                widths.append(img.size[0])
                heights.append(img.size[1])
                
            # B. Extract metadata ONLY if the image opened successfully
            file_names.append(img_path.name)
            class_labels.append(img_path.name.strip().split('_')[0])
            
        except Exception as e:
            # C. Exception handling for invalid data image file
            error_logs.append(f"<li style='color: red;'>Error reading {img_path.name}: {e}</li>")

    # Add error logs to HTML if any exist
    if error_logs:
        log_html += "".join(error_logs)
    else:
        log_html += "<li style='color: green;'>All images read successfully without errors.</li>"

    # Build the DataFrame Table of Consisting of File Name and its Classes
    df = pd.DataFrame({
        'fileName': file_names,
        'classLabel': class_labels
    })

    log_html += "<h4 style='text-align: left; margin-bottom: 5px;'>Class Labels Found</h4>"
    log_html += "<table style='margin: 0; text-align: left; border: none;'>"
    for i, label in enumerate(df['classLabel'].unique(), start=1):
        # Determine background color based on odd/even rows
        bg_color = "azure" if i % 2 == 0 else "paleturquoise" 
        
        # Apply the bg_color to the <tr> tag
        log_html += f"<tr style='background-color: {bg_color};'>\
            <td style='color: black; text-align: left; padding-right: 10px; border: none;'><strong>Class {i}</strong></td>\
            <td style='color: black; text-align: center; border: none;'>{label}</td></tr>"
    log_html += "</table>"

    # Calculate Medians of the data images 
    median_w, median_h = 0, 0
    log_html += "<h4 style='text-align: left; margin-top: 15px; margin-bottom: 5px;'>Dataset Statistics</h4>"
    
    if widths and heights:
        median_w = statistics.median(widths)
        median_h = statistics.median(heights)
        
        log_html += "<table style='margin: 0; text-align: left; border: none;'>"
        
        # Row 1 (Odd) - paleturquoise
        log_html += f"<tr style='background-color: paleturquoise;'>\
            <td style='color: black; text-align: left; padding-right: 10px; border: none;'><strong>Total Images Evaluated</strong></td>\
            <td style='color: black; text-align: center; border: none;'>{len(widths)}</td></tr>"
            
        # Row 2 (Even) - azure
        log_html += f"<tr style='background-color: azure;'>\
            <td style='color: black; text-align: left; padding-right: 10px; border: none;'><strong>Image Median Width</strong></td>\
            <td style='color: black; text-align: center; border: none;'>{median_w} px</td></tr>"
            
        # Row 3 (Odd) - paleturquoise
        log_html += f"<tr style='background-color: paleturquoise;'>\
            <td style='color: black; text-align: left; padding-right: 10px; border: none;'><strong>Image Median Height</strong></td>\
            <td style='color: black; text-align: center; border: none;'>{median_h} px</td></tr>"
            
        log_html += "</table>"
    else:
        log_html += "<div style='color: red; text-align: center;'>No valid .jpg images found.</div>"
        
    log_html += "<hr style='margin-top: 15px; border-top: 1px solid #ddd;'>"
        
    return df, median_w, median_h, log_html

# ---------------------------------------------------------- #
# Execute the Function for Both Directories
# ---------------------------------------------------------- #

# Define the path directories
base_dir = Path.cwd().parents[0]
img_train_dir = base_dir / "train"
img_test_dir = base_dir / "test"

# Process Training and Validation Datasets from 'train' directory
class_train_df, img_T_widths_median, img_T_heights_median, train_logs = process_image_directory(
    directory_path = img_train_dir, 
    dataset_name = "Training and Validation Dataset"
)

# Sampling of the training and validation dataframe for verification
train_html = (class_train_df.sample(min(5, len(class_train_df)))
        .style
        .set_properties(**{'text-align': 'center', 'color': 'black'})
        .set_table_styles([
            {'selector':'tr:nth-child(even)', 'props': [('background-color', 'azure')]},
            {'selector':'th', 'props':[('text-align', 'center'), ('color', 'black'), ('font-weight', 'bold')]},
            {'selector':'tr:nth-child(odd)', 'props':[('text-align', 'center'), ('background-color', 'paleturquoise')]}
        ])).to_html()

# Process Testing Datasets from 'test' directory
class_test_df, img_t_widths_median, img_t_heights_median, test_logs = process_image_directory(
    directory_path = img_test_dir, 
    dataset_name = "Testing Dataset"
)

# Sampling of the testing dataframe for verification
test_html = (class_test_df.sample(min(5, len(class_test_df)))
        .style
        .set_properties(**{'text-align': 'center', 'color': 'black'})
        .set_table_styles([
            {'selector':'tr:nth-child(even)', 'props': [('background-color', 'azure')]},
            {'selector':'th', 'props':[('text-align', 'center'), ('color', 'black'), ('font-weight', 'bold')]},
            {'selector':'tr:nth-child(odd)', 'props':[('text-align', 'center'), ('background-color', 'paleturquoise')]}
        ])).to_html()

# Wrapping them html in a Flex container to enable side-by-side display
train_test_html = f"""
<div style="display: flex; flex-direction: row; justify-content: space-around; align-items: flex-start; gap: 20px;">
    <div style="flex: 1; padding: 10px; border: 1px solid #ccc; border-radius: 16px;">
        <h3 style="text-align: center; color: 'black';">Training & Validation Dataset</h3>
            {train_logs}
        <h4 style="text-align: left;">DataFrame Sample</h4>
        <div style="display: flex; justify-content: left;">
            {train_html}
        </div>
    </div>
    
    <div style="flex: 1; padding: 10px; border: 1px solid #ccc; border-radius: 16px;">
        <h3 style="text-align: center; color: 'black';">Testing Dataset</h3>
            {test_logs}
        <h4 style="text-align: left;">DataFrame Sample</h4>
        <div style="display: flex; justify-content: left;">
            {test_html}
        </div>
    </div>
</div>
"""
display(HTML(train_test_html))

In [18]:
# ========================================================== #
# 4. Define a New Class: FruitLabeller and Supporting Functions
# Purpose: Create a custom PyTorch Dataset to load images, apply 
# transformations, and encode string labels to numeric indices 
# using a provided DataFrame. Includes a helper function to 
# visually summarize the dataset's class distribution.
# ========================================================== #

class FruitLabeller(Dataset):
    def __init__(self, class_dataframe, img_directory, transforms = None):
        self.dataframe = class_dataframe
        self.directory = img_directory
        self.transforms = transforms

        unique_labels = sorted(class_dataframe['classLabel'].unique())
        self.label_map = {label: idx for idx, label in enumerate(unique_labels)}
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx]['fileName']
        img_path = self.directory / img_name
        image = Image.open(img_path).convert("RGB")

        label_str = self.dataframe.iloc[idx]['classLabel']
        label = self.label_map[label_str]

        if self.transforms:
            image = self.transforms(image)
        
        return image, label
    
def display_dataset_shape(df, dataset_name):
    shape_df = df['classLabel'].value_counts().sort_index().reset_index()
    shape_df.columns = ['Class Label', 'Total Image']

    display(HTML(f"<h3>{dataset_name} Dataset Shape:</h3>"))
    display(shape_df.style
            .set_properties(**{'color':'white', 'text-align':'center'})
            .set_table_styles([
            {'selector':'tr:nth-child(even)', 'props': [('background-color', '#708090')]},
            {'selector':'th', 'props':[('text-align', 'center')]},
            ])
            .hide(axis="index")
            )